# Train RF-DETR Nano on COCO2017 (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/train-coco2017.ipynb)

Downloads the full public **COCO2017** dataset and trains **RF-DETR Nano** for 40 epochs, end to end, on a single
free Colab GPU. Every `TrainConfig` field besides the dataset location, epoch count, and output directory is left
at its shipped default — the point of this notebook is that the defaults are fast enough on their own.

Kept deliberately minimal: download, train, plot the metrics `CSVLogger` wrote during training. For dataset
preview, checkpoint saving, and inference visualization, see `fine-tune_detection.ipynb`.

## Why this notebook is simple on purpose

Recent releases moved several training-throughput fixes directly into the default configuration, so a stock run
picks them up automatically:

- **Validation forwards one model per epoch, not two.** The base-model forward pass used to run alongside the EMA
  forward every validation batch; it is now skipped when `use_ema=True` (the default).
- **`grad_accum_steps` now defaults to `1`** instead of `4`. Gradient accumulation is an explicit opt-in — raise
  `batch_size` for your GPU first, and reach for accumulation only when memory forces a smaller physical batch.
  Measured on one L4: `batch_size=16, grad_accum_steps=1` ran 27% faster per epoch than `batch_size=4,
  grad_accum_steps=4` at the same nominal effective batch, with equal mAP.
- **`eval_batch_size`** decouples the validation/test dataloaders from the training micro-batch size, so a small
  training batch no longer forces small (slower) evaluation batches.
- **COCO mAP computation reads each image's detection scores once instead of once per detection.**
- **The transformer skips materializing tensors it would otherwise reuse unchanged** on the single-feature-level
  path that Nano (and every current detection size) uses by default.
- **Pre-training sanity-check validation is skipped by default.**

None of this needs a flag — it is what `RFDETRNano()` and `TrainConfig` already do. This notebook sets only
`dataset_file`, `dataset_dir`, `output_dir`, `epochs`, and `batch_size="auto"` (so it adapts to whatever GPU Colab
hands you), and turns off TensorBoard logging so the notebook does not require the `loggers` extra.

## Scope and honest expectations

COCO2017 has 118,287 training images and 5,000 validation images — two to three orders of magnitude larger than
the small Roboflow Universe datasets in the other fine-tuning cookbooks. There is no COCO2017-scale timing
measurement in this repository yet, so no fixed "N minutes per epoch" number is given here. **Time your own first
epoch** (the progress bar prints per-epoch elapsed time) before committing GPU time to all 40. As a rough anchor:
an internal L4 measurement on a ~6,900-image dataset with `rfdetr-small` at batch 16 took ~324 s/epoch
(train + validation); COCO2017 has about 17x more training images and 2.5x more validation images, and
`rfdetr-nano` is smaller and runs at a lower resolution than `rfdetr-small`, which pulls the other way. Expect the
full 40-epoch run to take multiple hours even on a fast GPU.

> **Free Colab sessions are not guaranteed to stay connected long enough for a single 40-epoch run.** Training
> checkpoints every `checkpoint_interval` epochs (10 by default) to `last.ckpt`, so if a session disconnects,
> reopen the notebook, set `train_config.resume` to that file's path, and re-run the training cell to continue.
> A100-class hardware (e.g. Colab Pro) comfortably fits the full run in one session; a free T4/L4 may not.

COCO2017 also needs about 19 GB of disk once downloaded (the archives are deleted right after extraction to avoid
a ~38 GB peak). Confirm your Colab runtime has that much free space before starting the download.

## Setup

The training-throughput work described above has not shipped in a tagged release yet, so this cell installs from
the `develop` branch. Once a release containing it is out, replace this with
`pip install -q "rfdetr[train,visual]>=1.10.0"`.

**GPU required.** `batch_size="auto"` (used below) probes CUDA memory directly, so a GPU runtime is mandatory —
in Colab: **Runtime → Change runtime type → T4 GPU** (or better, if available on your plan).

In [ ]:
!pip install -q "rfdetr[train,visual] @ git+https://github.com/roboflow/rf-detr.git@develop"

## 1 - Download COCO2017

Plain `wget`/`unzip` into the standard COCO layout RF-DETR's `dataset_file="coco"` loader expects:
`coco2017/train2017/`, `coco2017/val2017/`, `coco2017/annotations/instances_{train,val}2017.json`.

`wget -c` resumes an interrupted download instead of restarting it; `unzip -n` never overwrites a file already
extracted. Both make re-running this cell after a disconnected Colab session safe. The `rm` at the end deletes the
archives once extracted, to keep peak disk usage down.

In [ ]:
!mkdir -p datasets/coco2017
!wget -c -q --show-progress http://images.cocodataset.org/zips/train2017.zip -P datasets/coco2017
!wget -c -q --show-progress http://images.cocodataset.org/zips/val2017.zip -P datasets/coco2017
!wget -c -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P datasets/coco2017
!unzip -n -q datasets/coco2017/train2017.zip -d datasets/coco2017
!unzip -n -q datasets/coco2017/val2017.zip -d datasets/coco2017
!unzip -n -q datasets/coco2017/annotations_trainval2017.zip -d datasets/coco2017
!rm -f datasets/coco2017/train2017.zip datasets/coco2017/val2017.zip datasets/coco2017/annotations_trainval2017.zip

## 2 - Configure training

`RFDETRNano()` loads the released COCO-pretrained Nano checkpoint by default — this notebook continues training
from it rather than from random initialization. Training from scratch is not the recommended path for RF-DETR;
continuing from the pretrained checkpoint on the same dataset it was pretrained on still exercises the full
40-epoch training loop and every optimization listed above, which is what this notebook demonstrates.

`num_classes` is left at its default of `90` — the standard COCO category-id space RF-DETR's `dataset_file="coco"`
loader and the pretrained checkpoint both already use, so it does not need to be passed explicitly.

`batch_size="auto"` probes the GPU for the largest safe batch and sets `grad_accum_steps` to match, so this same
cell adapts to a T4, L4, or A100 Colab runtime without editing any numbers by hand.

In [ ]:
import torch

from rfdetr import RFDETRNano
from rfdetr.config import TrainConfig
from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
from rfdetr.utilities.reproducibility import seed_all
from rfdetr.visualize.training import plot_loss_metrics, plot_map_metrics

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook requires a CUDA GPU (batch_size='auto' probes CUDA memory directly). "
        "In Colab: Runtime -> Change runtime type -> GPU."
    )

COCO_ROOT = "datasets/coco2017"
OUTPUT_DIR = "output/det_coco2017_nano"
EPOCHS = 40

seed_all(0)
variant = RFDETRNano()  # type: ignore[no-untyped-call]

train_config = TrainConfig(
    dataset_file="coco",
    dataset_dir=COCO_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size="auto",
    tensorboard=False,
    progress_bar="tqdm",
)

datamodule = RFDETRDataModule(variant.model_config, train_config)
model = RFDETRModelModule(variant.model_config, train_config)
trainer = build_trainer(train_config, variant.model_config)

## 3 - Train

`trainer.fit` hands control to PyTorch Lightning for the full loop: forward pass, bipartite matching loss,
gradient accumulation, weight updates, learning-rate scheduling, and periodic COCO mAP validation. `CSVLogger`
appends one row of metrics per epoch to `output_dir/metrics.csv`, plotted below. The progress bar reports elapsed
time per epoch — use the first completed epoch to size how long the remaining 39 will take on your GPU before
deciding whether to let this run uninterrupted or resume it across multiple sessions.

To **resume an interrupted run**, set `train_config.resume` to `f"{OUTPUT_DIR}/last.ckpt"` and re-run this cell.

In [ ]:
trainer.fit(model, datamodule=datamodule, ckpt_path=train_config.resume or None)

## 4 - Plot CSVLogger metrics

`bbox/map` is the primary metric — standard COCO bounding-box mAP averaged across IoU thresholds 0.50-0.95.
`bbox/map_50` rises fastest and is the clearest early signal of whether training is progressing at all;
`bbox/map_75` reflects localization precision, not just whether objects are found.

In [ ]:
from IPython.display import display
from matplotlib import pyplot as plt

METRICS_CSV = f"{OUTPUT_DIR}/metrics.csv"

loss_figure = plot_loss_metrics(METRICS_CSV)
display(loss_figure)
plt.close(loss_figure)

In [ ]:
map_figure = plot_map_metrics(METRICS_CSV)
display(map_figure)
plt.close(map_figure)